# Lab4.4 — Detecția muchiilor cu Sobel

Operatorul Sobel estimează gradientul intensității imaginii pe direcțiile X și Y. Este un exemplu bun de operație locală, repetată pentru fiecare pixel, deci potrivită pentru paralelizare hardware.


## Obiective

- aplicarea operatorului Sobel X și Y;
- calculul magnitudinii gradientului;
- vizualizarea muchiilor;
- măsurarea timpului de execuție software;
- identificarea potențialului de accelerare hardware.


In [ ]:
from pathlib import Path

IMAGE_PATH = Path("images/fruits.jpg")

if not IMAGE_PATH.exists():
    raise FileNotFoundError(
        "Nu a fost găsită imaginea Lab4/images/fruits.jpg. "
        "Verificați că repository-ul a fost copiat complet pe placă."
    )

print("Image:", IMAGE_PATH)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time

image_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
if image_bgr is None:
    raise RuntimeError("Imaginea nu a putut fi încărcată.")

gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

## 1. Gradientul pe X și Y

Folosim un nucleu Sobel de dimensiune 3 × 3 și rezultate intermediare `float32`.


In [ ]:
gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)

print("gx:", gx.shape, gx.dtype)
print("gy:", gy.shape, gy.dtype)

## 2. Magnitudinea gradientului

Calculăm:

```text
G = sqrt(Gx² + Gy²)
```


In [ ]:
magnitude = cv2.magnitude(gx, gy)
edges = cv2.convertScaleAbs(magnitude)

print("Magnitude dtype:", magnitude.dtype)
print("Edges dtype    :", edges.dtype)

In [ ]:
plt.figure(figsize=(9, 5))
plt.imshow(edges, cmap="gray")
plt.title("Sobel edge magnitude")
plt.axis("off")
plt.show()

## 3. Funcție software Sobel


In [ ]:
def sobel_software(gray_image):
    gx = cv2.Sobel(gray_image, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray_image, cv2.CV_32F, 0, 1, ksize=3)
    magnitude = cv2.magnitude(gx, gy)
    return cv2.convertScaleAbs(magnitude)

## 4. Benchmark Sobel


In [ ]:
REPEATS = 30

sobel_software(gray)
t0 = time.perf_counter()
for _ in range(REPEATS):
    tmp = sobel_software(gray)
mean_t = (time.perf_counter() - t0) / REPEATS

print(f"Sobel: {mean_t*1e3:.3f} ms")
print(f"FPS  : {1/mean_t:.2f}")

## 5. Influența rezoluției

Pregătim imagini grayscale la trei rezoluții; resize-ul folosit pentru pregătire nu este inclus în măsurarea Sobel.


In [ ]:
for w, h in ((320, 240), (640, 480), (1280, 720)):
    test_gray = cv2.resize(gray, (w, h), interpolation=cv2.INTER_LINEAR)
    sobel_software(test_gray)

    t0 = time.perf_counter()
    for _ in range(20):
        tmp = sobel_software(test_gray)
    mean_t = (time.perf_counter() - t0) / 20

    print(f"{w:4d}x{h:<4d}: {mean_t*1e3:8.3f} ms  -> {1/mean_t:7.2f} FPS")

## 6. Exercițiu

Pentru rezoluția **640 × 480**, măsurați separat:

- timpul pentru `Sobel X`;
- timpul pentru `Sobel Y`;
- timpul pentru calculul magnitudinii;
- timpul total.

Identificați etapa dominantă.


In [ ]:
# Scrieți aici soluția


## 7. Întrebări

1. Ce informație reprezintă gradientul pe X și gradientul pe Y?
2. De ce sunt folosite rezultate intermediare cu precizie mai mare decât `uint8`?
3. De ce Sobel se pretează la o implementare pipeline în FPGA?
